# 3. Featurizing the data

## 3.1.1 Gathering data

In [6]:
import pickle 
import numpy as np 
import pandas as pd 
import gensim
import nltk; nltk.download("punkt")
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from transformers import AutoTokenizer, AutoModel
import torch
import tensorflow as tf
import tensorflow_hub as hub
from tqdm import tqdm 

[nltk_data] Downloading package punkt to /Users/rathish/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [7]:
dataloc ='/Users/rathish/Documents/Projects/Opportunity_Application_Ranker/data'
fdata = pd.read_pickle(dataloc + '/cleaned_data/preprocesseddata.pkl')
fdata.head()

,OpportunityId,ApplicationId,ExternalBriefDescription,ExternalDescription,Title,JobCategoryName,IsRejected,IsCandidateInternal,BehaviorCriteria,MotivationCriteria,...,SkillCriteria__usepp,WorkExperiences__usepp,Educations__usepp,LicenseAndCertifications__usepp,Skills__usepp,Motivations__usepp,Behaviors__usepp,StepId__usepp,StepName__usepp,StepGroup__usepp
0,MbzeABKVn06G8irkoHJeIg==,nTzdqGj020CYqTouPocGSg==,"$16.00 Per Hour\n\nAt Orkin, our purpose is to...",<p><strong>$16.00 Per Hour</strong></p>\n<p><s...,Customer Service Specialist,Customer Service,True,False,[{'Description': 'Capable of carrying out a gi...,[{'Description': 'Inspired to perform well by ...,...,MinimumScaleValue 3 MinimumScaleValueName Inte...,EndMonth None EndYear None JobTitle Call Cente...,Degree Some college Description None Graduatio...,None,ScaleValue 4 ScaleValueName Advanced Skill Clo...,Description Inspired to perform well by moneta...,Description Devoted to a task or purpose with ...,K8yQlic+/UiXxBMpOnAoLQ==,Decline,declined
1,7SPt0A57/kyzM9hE9vxDRg==,QVk5MFCZ70WvlZE9FzAW9g==,"$15.00 Per Hour\n\nAt Orkin, our purpose is to...",<p><strong>$15.00 Per Hour</strong></p>\n<p><s...,Customer Service Specialist,Customer Service,True,False,[{'Description': 'Capable of carrying out a gi...,[{'Description': 'Inspired to perform well by ...,...,MinimumScaleValue 3 MinimumScaleValueName Inte...,EndMonth None EndYear None JobTitle Coordinato...,Degree Diploma Description None GraduationMont...,None,ScaleValue 5 ScaleValueName Expert Skill Sales,None,None,K8yQlic+/UiXxBMpOnAoLQ==,Decline,declined
2,7SPt0A57/kyzM9hE9vxDRg==,I1kcPlAw3E+rqceh0qrutQ==,"$15.00 Per Hour\n\nAt Orkin, our purpose is to...",<p><strong>$15.00 Per Hour</strong></p>\n<p><s...,Customer Service Specialist,Customer Service,True,False,[{'Description': 'Capable of carrying out a gi...,[{'Description': 'Inspired to perform well by ...,...,MinimumScaleValue 3 MinimumScaleValueName Inte...,EndMonth None EndYear None JobTitle Direct Car...,Degree HIGH SCHOOL DIPLOMA Description None Gr...,None,ScaleValue 4 ScaleValueName Advanced Skill Cash,None,None,K8yQlic+/UiXxBMpOnAoLQ==,Decline,declined
3,zolSWBFjWESbfkj8AXLYwA==,VTCXZK6/ZUWJDpxTcm2CRg==,"$15.00 Per Hour\n\nAt Orkin, our purpose is to...",<p><strong>$15.00 Per Hour</strong></p>\n<p><s...,Customer Service Specialist,Customer Service,True,False,[{'Description': 'Capable of carrying out a gi...,[{'Description': 'Inspired to perform well by ...,...,MinimumScaleValue 3 MinimumScaleValueName Inte...,EndMonth None EndYear 2019.0 JobTitle Package ...,Degree Associate in Early Description None Gra...,None,ScaleValue 5 ScaleValueName Expert Skill Cashier,None,None,K8yQlic+/UiXxBMpOnAoLQ==,Decline,declined
4,zolSWBFjWESbfkj8AXLYwA==,I6KgcL0jdkG8wBnL1+BZ/g==,"$15.00 Per Hour\n\nAt Orkin, our purpose is to...",<p><strong>$15.00 Per Hour</strong></p>\n<p><s...,Customer Service Specialist,Customer Service,True,False,[{'Description': 'Capable of carrying out a gi...,[{'Description': 'Inspired to perform well by ...,...,MinimumScaleValue 3 MinimumScaleValueName Inte...,EndMonth None EndYear None JobTitle Warehouse ...,Degree Bachelor of Business Admin Description ...,None,ScaleValue 5 ScaleValueName Expert Skill Forklift,None,None,K8yQlic+/UiXxBMpOnAoLQ==,Decline,declined


### 3.1.2 Defining column names for featurization 

In [8]:
# Defining list containing names of the columns

job_column = [
    'ExternalBriefDescription',
    'ExternalDescription', 
    'Title', 
    'JobCategoryName'
]

uid_column = ['OpportunityId', 'ApplicationId']

# Column - 'Tag' Will be added later
can_column = [
    'IsCandidateInternal',
    'BehaviorCriteria', 
    'MotivationCriteria',
    'EducationCriteria', 
    'LicenseAndCertificationCriteria', 
    'SkillCriteria', 
    'WorkExperiences', 
    'Educations', 
    'LicenseAndCertifications', 
    'Skills', 
    'Motivations', 
    'Behaviors', 
    'StepId', 
    'StepName', 
    'StepGroup',
    'pass_first_step'
] 

sel_column = ['IsRejected']

# Defining list of columns based on the type of contents

str_column = [
    'ExternalBriefDescription', 
    'ExternalDescription', 
    'Title', 
    'JobCategoryName', 
    'BehaviorCriteria', 
    'MotivationCriteria', 
    'EducationCriteria', 
    'LicenseAndCertificationCriteria', 
    'SkillCriteria', 
    'WorkExperiences', 
    'Educations', 
    'LicenseAndCertifications', 
    'Skills', 
    'Motivations', 
    'Behaviors', 
    'StepId', 
    'StepName', 
    'StepGroup'
]

bool_column = ['IsCandidateInternal', 'pass_first_step']

float_column = ['Tag']

## 3.2 TFIDF weighted word2vec vectorization

Let's use tfidf weighted word2vec for generating the necessary features to measure similarity. 
To do this, let's generate two functions - tfidf_weighted_word2vec, which inputs the column name and the data to generate the tfidf vocabulary, tfidf - idf values, and word2vec vocab:vector after training the data. 
The second function i.e.tfidfw2v_vectorizer intakes all of the variables generated in the first function along and applies it to a particular text to achieve tfidf weighted word to vec. 

### 3.2.1 Creating functions that derive tfidf weighted word2vec information

In [9]:

def tfidf_weighted_word2vec(data, colname):
    '''
    Function calculates the tfidf (from scikit-learn's TfidfVectorizer) 
    weighted word2vec (from gensim.Word2Vec) as per the following formulae:
    Tfidf w2v (w1,w2..) = 
    (tfidf(w1) * w2v(w1) + tfidf(w2) * w2v(w2) + …)/(tfidf(w1) + tfidf(w2) + …)
    
    Args:
    data : pandas.DataFrame
    Dataset containing columns text for converting into word2vec
    col_names : str
    Name of the target column on which the operation needs to be performed

    Returns: 
     w2v : dict
    Dictionary with keys value pairs containing words and the respective vector
        word2weight : dict 
    vocab(dict_item):
    '''

    # Generating coldata
    c = data[colname].tolist()
    c = [str(x) for x in c]
    coldata = []
    
    # Creating model and tokenizing words for the moedl
    model = Word2Vec(window = 2, min_count = 3, sg = 1, vector_size = 100)
    
    for x in c:
        coldata.append(gensim.utils.simple_preprocess(x))

    # Creating model vocabulary from the tokens and then training and 
    # later bundling into a dictionary
    
    model.build_vocab(coldata)
    model.train(corpus_iterable=coldata, total_examples= model.corpus_count, 
                epochs=model.epochs)
    
    w2v = dict(zip(model.wv.index_to_key, model.wv.vectors.round(3)))
    
    # Creating tfidf vectorizer model
    
    tfidfvectorizer = TfidfVectorizer()
    tfidfvectorizer.fit_transform(c)
    vocab = tfidfvectorizer.vocabulary_.items()

    # Generating word2weight dictionary of word and its tfidf values
    word2weight = [(w, round(tfidfvectorizer.idf_[i], 3)) 
                   for w, i in tfidfvectorizer.vocabulary_.items()]
    word2weight  = dict(word2weight)

    return w2v, word2weight, vocab

def tfidfw2v_vectorizer(text, w2v, word2weight):
    '''
    Function intakes a text along with word and its idf_ vector, and wor

    Args:
    d : text
    '''
    words = text.split() 

    if len(words) == 0:
        
        return np.zeros(100)

    else:
        numerator_vector = np.zeros(100)
        denominator_value = 0.0
        
        for word in words:
            
            if word in w2v.keys() and word in word2weight.keys():
                
                numerator_val = words.count(word)*word2weight[word]*w2v[word]
                numerator_vector += numerator_val
               
                denominator_val = words.count(word)*word2weight[word]
                denominator_value += denominator_val
        
        if denominator_value == 0.0:
            
            return np.zeros(100)
       
        else: 
            
            return np.round(numerator_vector/denominator_value, 3)


### 3.2.2 Applying the TFIDF weighted word2vec function

In [10]:
# Applying the tfidf - avg word 2 vec

for colname in str_column:
    w2v, word2weight, vocab = tfidf_weighted_word2vec(
        fdata, colname + "__w2vpp"
    )
    
    fdata[colname + "__w2v"] = fdata[colname + "__w2vpp"].apply(
        lambda x: tfidfw2v_vectorizer(x, w2v,word2weight)
    )

In [11]:
print(word2weight)

{'declin': 1.654, 'interview': 2.029, 'background': 5.041, 'check': 5.041, 'hire': 5.344, 'screen': 6.622, 'refer': 8.622}


### 3.2.3 Handling boolean float columns with one hot encoding

In [12]:
# Applying OneHotEncoder

onehotencoder = OneHotEncoder(sparse_output = False, handle_unknown = 'ignore')

for colname in bool_column:
    fdata[colname + "__w2v"] = list(onehotencoder.fit_transform(
        np.reshape(np.array(fdata[colname]), (-1, 1))
        )
    ) 

'''minmaxscaler = MinMaxScaler() isn't being applied as float data contained
None objects. The None objects were converted to -1. Applying minmax scaller 
would modify the data'''

for colname in float_column:
    fdata[colname + "__w2v"] = list(np.reshape(
        np.array(fdata[colname]), (-1, 1)
        )
    )

### 3.2.4 Stacking all the vectors together

In [13]:
# Function for adding relevant arrays column wise to create large vector
def hstacker(row_arrays):
    return np.concatenate(row_arrays)

In [14]:
# Gathering/concatenating the opportunity/job related vectors into a new column
fdata['opportunity__w2v'] = fdata[[m + "__w2v" for m in job_column]].apply(
    hstacker, axis = 1
)

# Gathering/Concatenating the candidate related vectors into a new column
fdata['candidate__w2v'] = fdata[[m + "__w2v" for m in can_column]].apply(
    hstacker, axis = 1
)

# Adding column 'Tag' to the candidate_w2v as it contains float values

fdata['candidate__w2v'] = fdata[['candidate__w2v', 'Tag__w2v']].apply(
    hstacker, axis = 1
)

/var/folders/_0/gwtrw0ms65zdzng2095bbrdr0000gn/T/ipykernel_16606/3257446904.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fdata['candidate__w2v'] = fdata[[m + "__w2v" for m in can_column]].apply(


## 3.3 Generating embeddings using transformer based large language models

### 3.3.1 Gathering necessary data for BERT based models

In [15]:
# Gathering necessary columns

fdata['opportunity__str'] = fdata[[m + "__bertpp" 
                                   for m in job_column if m in str_column]
                                   ].agg(" ".join, axis = 1).tolist()

fdata['candidate__str'] = fdata[[m + "__bertpp" 
                                 for m in can_column if m in str_column]
                                 ].agg( " ".join, axis = 1).tolist()


# Applying OneHotEncoder for boolean columns

onehotencoder = OneHotEncoder(sparse_output = False, handle_unknown = 'ignore')

for colname in bool_column:
    fdata[colname + "__bert"] = list(onehotencoder.fit_transform(
        np.reshape(np.array(fdata[colname]), (-1, 1))
        )
    ) 
    

# Gathering floatcolumn

for colname in float_column:
    fdata[colname + "__bert"] = list(np.reshape(
        np.array(fdata[colname]), (-1, 1)))

/var/folders/_0/gwtrw0ms65zdzng2095bbrdr0000gn/T/ipykernel_16606/1129281344.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fdata['opportunity__str'] = fdata[[m + "__bertpp"
/var/folders/_0/gwtrw0ms65zdzng2095bbrdr0000gn/T/ipykernel_16606/1129281344.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fdata['candidate__str'] = fdata[[m + "__bertpp"
/var/folders/_0/gwtrw0ms65zdzng2095bbrdr0000gn/T/ipykernel_16606/1129281344.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame

It was observed in our EDA that there were only 8473 opportunities, which implies the text in the string columns are likely to be repeated for same jobs. Also, even though the the applicaiton ID is unique, it is likely that these same candidate with same candidate details may have applied for multiple jobs. In order to reduce the time and space capacities, lets check if the string columns when put together contains any duplicates. 

In [16]:


job_data = fdata[job_column + ["OpportunityId"]].duplicated()

 # ApplicationID ommitted as it was unique for all values

can_data = fdata['candidate__str'].duplicated()

print(job_data.value_counts(),  can_data.value_counts())
fdata.shape[0]

True     101794
False      8473
Name: count, dtype: int64 candidate__str
False    109666
True        601
Name: count, dtype: int64


110267

With so many (about 90%) duplicate values in candidate opportunities, it is prudent to therefore pass non-duplicates to the BERT and then associate the embeddings back to data.

There wouldn't be significant time improvements(a reduction around 0.6% in the reduction of values) when pass non-duplicates to the BERT in case of the candidate textual information, so we do not drop duplicates, in this case. 

In [17]:
llm_job_data_bert = fdata[["OpportunityId"] + 
                          [m + "__bertpp" 
                           for m in job_column 
                           if m in str_column]].drop_duplicates()

# We are not dropping duplicates for the combined data containing ApplicationId

llm_candidate_data_bert = fdata[["ApplicationId"] + 
                                [m + "__bertpp" 
                                 for m in can_column 
                                 if m in str_column]]  

In [18]:
# Loading BERT model and tokenizer

# Shifting to GPU for faster calculations

device = (
    torch.device("mps") 
    if torch.backends.mps.is_available() 
    else torch.device("cpu")
)

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Moving model to GPU
model = AutoModel.from_pretrained(model_name).to(device)  

In [19]:
# Gathering BERT base embedded vectors for job opportunity columns
nltk.download("punkt")
job_opportunityid_bert_dict = {}

for index, row in tqdm(
    llm_job_data_bert[
        [m + "__bertpp" for m in job_column if m in str_column]
    ].iterrows(), 
    desc = "Processing rows",  
    total = len(llm_job_data_bert)
):
    
    embeddings_values = []

    for column in [m for m in job_column if m in str_column]:
        text = llm_job_data_bert.at[index, column + "__bertpp"]
        sentences = nltk.sent_tokenize(text)
        
        input = tokenizer(
            sentences, padding = True, 
            truncation = True, 
            return_tensors = "pt"
        ).to(device) 
        
        with torch.no_grad():
            output = model(**input)
        
        embeddings_values.append(
            np.mean(
                output.last_hidden_state.mean(dim=1).cpu().numpy(), axis = 0
                )
            )
        
    vector = np.hstack(tuple(embeddings_values))
    job_opportunityid_bert_dict[
        llm_job_data_bert.at[index, "OpportunityId"]
        ] = vector 

[nltk_data] Downloading package punkt to /Users/rathish/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Processing rows:  18%|█▊        | 1541/8473 [29:14<3:55:23,  2.04s/it] 

In [ ]:
# Gathering BERT base embedded vector for candidate columns

can_applicationid_bert_dict = {}

for index, row in tqdm(
    llm_candidate_data_bert[
        [m + "__bertpp" for m in can_column if m in str_column]
    ].iterrows(), 
        desc = "Processing rows", 
        total = len(llm_candidate_data_bert)
    ):
    
    embeddings_values = []
    
    for column in [m for m in can_column if m in str_column]:
        text = llm_candidate_data_bert.at[index, column + "__bertpp"]
        sentences = nltk.sent_tokenize(text)
        
        input = tokenizer(
            sentences, padding = True, 
            truncation = True, 
            return_tensors = "pt"
        ).to(device) 
        
        with torch.no_grad():
            output = model(**input)
        embeddings_values.append(
            np.mean(
                output.last_hidden_state.mean(dim=1).cpu().numpy(), axis = 0))
            
    vector = np.hstack(tuple(embeddings_values))
    can_applicationid_bert_dict[
        llm_candidate_data_bert.at[index, "ApplicationId"]
    ] = vector 

Processing rows:   0%|          | 0/110267 [00:00<?, ?it/s]

Processing rows:   0%|          | 6/110267 [00:08<44:55:43,  1.47s/it]


KeyboardInterrupt: 

In [ ]:
print(can_applicationid_bert_dict['nTzdqGj020CYqTouPocGSg=='].shape, 
      job_opportunityid_bert_dict['MbzeABKVn06G8irkoHJeIg=='].shape)

(10752,) (3072,)


In [ ]:
# Applying vectors to the data 

for column_name in fdata[uid_column]:
    if column_name == "ApplicationId":
        fdata["opportunity__bert"] = fdata["ApplicationId"].apply(
            lambda x : can_applicationid_bert_dict[x]
        )
        
    if column_name == "OpportunityId":
        fdata["candidate__bert"] = fdata["OpportunityId"].apply(
            lambda x : job_opportunityid_bert_dict[x]
        )

In [ ]:
fdata[["opportunity__bert", "candidate__bert"]]

print(fdata['opportunity__bert'][1].shape, fdata['candidate__bert'][1].shape)

NameError: name 'fdata' is not defined

# Using Universal Sentence Encoder Embeddings

In [ ]:
# Gathering data for USE based word embeddings 

llm_job_data_use = fdata[["OpportunityId"] + 
                         [m + "__usepp" 
                          for m in job_column 
                          if m in str_column
                          ]
                        ].drop_duplicates()

# We are not dropping duplicates for the combined data containing ApplicationI

llm_candidate_data_use = fdata[["ApplicationId"] + 
                               [m + "__usepp" 
                                for m in can_column 
                                if m in str_column
                                ]
                            ] 

In [ ]:
# Load the Universal Sentence Encoder module

url = "https://tfhub.dev/google/universal-sentence-encoder/4"
use_model = hub.load(url)

2023-11-13 14:23:52.915716: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2023-11-13 14:23:56.044259: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


In [ ]:
# Getting information on devices available in the local system:

tf.config.list_physical_devices()

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'),
 PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [ ]:
job_opportunityid_use_dict = {}

for index, row in tqdm(
    llm_job_data_use[
        [m + "__usepp" for m in job_column if m in str_column]
    ].iterrows(), 
    desc = "Processing rows", 
    total = len(llm_job_data_use)
):

    embeddings_values = []
    
    for column in [m for m in job_column if m in str_column]:
        text = llm_job_data_use.at[index, column + "__usepp"]
        sentences = nltk.sent_tokenize(text)

        embeddings_values.append(
            tf.reduce_mean(use_model(sentences), axis = 0).numpy()
        )
            
    vector = np.hstack(tuple(embeddings_values))
    job_opportunityid_use_dict[
        llm_job_data_use.at[index, "OpportunityId"]
    ] = vector 
    

Processing rows:   3%|▎         | 273/8473 [02:18<1:09:21,  1.97it/s]


KeyboardInterrupt: 

In [ ]:

can_applicationid_use_dict = {}

for index, row in tqdm(
    llm_candidate_data_use[
        [m + "__usepp" for m in can_column if m in str_column]
    ].iterrows(), 
    desc = "Processing rows", 
    total = len(llm_candidate_data_use)):

    embeddings_values = []
    for column in [m for m in can_column if m in str_column]:
        text = llm_candidate_data_use.at[index, column + "__usepp"]
        sentences = nltk.sent_tokenize(text)
        
        embeddings_values.append(
            tf.reduce_mean(use_model(sentences), axis = 0).numpy()
        )
            
    vector = np.hstack(tuple(embeddings_values))
    can_applicationid_use_dict[
        llm_candidate_data_use.at[index, "ApplicationId"]
    ] = vector 

Processing rows:   0%|          | 0/110267 [00:03<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# Applying vectors to the data 

for column_name in fdata[uid_column]:

    if column_name == "ApplicationId":
        fdata["opportunity__use"] = fdata["ApplicationId"].apply(
            lambda x : can_applicationid_use_dict[x]
        )
        
    if column_name == "OpportunityId":
        fdata["candidate__use"] = fdata["OpportunityId"].apply(
            lambda x : job_opportunityid_use_dict[x]
        )

KeyError: 'ajdzD/i3EkCeBu1VDltNtg=='

## Saving data for futher analysis

In [ ]:
# Creating a new list of model names

model_names = ["w2v", "bert", "use"]

# Exporting rdata to a pickle file

fdata[
    ["opportunity__" + m for m in model_names] + 
    ["candidate__" + m for m in model_names]].to_pickle(
        dataloc + "/cleaned_data/featurizeddata.pkl"
      )

# Saving large language model derived dictionaries. 
with open(dataloc + '/cleaned_data/variables.pkl', 'wb') as file:
    pickle.dump(job_opportunityid_bert_dict, file)
    pickle.dump(can_applicationid_bert_dict, file)
    pickle.dump(job_opportunityid_use_dict, file)
    pickle.dump(can_applicationid_use_dict, file)

KeyError: "['opportunity__bert', 'opportunity__use', 'candidate__bert', 'candidate__use'] not in index"